# Lakehouse Pipeline - Exploratory Analysis

This notebook provides interactive exploration of the e-commerce lakehouse data across three storage layers:
- **Snowflake MARTS schema** — dbt-produced fact and dimension tables (fct_orders, mart_conversion_funnel, mart_cohort_retention, dim_users)
- **Delta Lake on S3/MinIO** — Spark-produced curated aggregations (product_performance)
- **User features** — RFM segments and LTV estimates from the Ray/Spark feature engineering pipeline

Set environment variables in `.env` before running (see `.env.example`).

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import pyarrow.dataset as ds
import seaborn as sns
import snowflake.connector
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    database=os.environ.get("SNOWFLAKE_DATABASE", "LAKEHOUSE"),
    schema="MARTS",
    warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE", "COMPUTE_WH"),
    role=os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN"),
)

def sf_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    cols = [c[0].lower() for c in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

print("Snowflake connection established")

In [ ]:
orders = sf_query("""
    SELECT
        order_id,
        user_id,
        ordered_at::DATE        AS order_date,
        net_amount,
        payment_method,
        order_status
    FROM fct_orders
    WHERE ordered_at >= DATEADD('day', -90, CURRENT_DATE)
    ORDER BY ordered_at DESC
""")

print(f"Loaded {len(orders):,} orders from the last 90 days")
display(orders.describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(orders["net_amount"], bins=60, edgecolor="white", color="steelblue")
axes[0].set_title("Order Amount Distribution (90d)")
axes[0].set_xlabel("Net Amount ($)")
axes[0].set_ylabel("Count")

daily_revenue = orders.groupby("order_date")["net_amount"].sum().reset_index()
axes[1].plot(daily_revenue["order_date"], daily_revenue["net_amount"], linewidth=1.5, color="steelblue")
axes[1].set_title("Daily Revenue (90d)")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Revenue ($)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print("\nTop 5 payment methods by volume:")
display(orders["payment_method"].value_counts().head())

## Conversion Funnel Analysis

The `mart_conversion_funnel` table is built by a dbt model that aggregates daily session events from the Iceberg raw zone. The funnel shows aggregate drop-off across the seven canonical e-commerce steps.

In [ ]:
funnel = sf_query("""
    SELECT
        date,
        total_sessions,
        sessions_with_product_view,
        sessions_with_search,
        sessions_with_add_to_cart,
        sessions_with_checkout_start,
        sessions_with_purchase,
        overall_conversion_rate,
        browse_to_cart_rate,
        cart_to_purchase_rate
    FROM mart_conversion_funnel
    ORDER BY date DESC
    LIMIT 30
""")

print(f"Latest conversion rate: {funnel['overall_conversion_rate'].iloc[0]:.2%}")

latest = funnel.iloc[0]
funnel_steps = [
    ("Sessions",         int(latest["total_sessions"])),
    ("Product Views",    int(latest["sessions_with_product_view"])),
    ("Searches",         int(latest["sessions_with_search"])),
    ("Add to Cart",      int(latest["sessions_with_add_to_cart"])),
    ("Checkout Start",   int(latest["sessions_with_checkout_start"])),
    ("Purchase",         int(latest["sessions_with_purchase"])),
]
funnel_df = pd.DataFrame(funnel_steps, columns=["step", "users"])

fig = px.funnel(
    funnel_df,
    x="users",
    y="step",
    title=f"Conversion Funnel — {latest['date']}",
    color_discrete_sequence=px.colors.sequential.Blues_r,
)
fig.update_layout(height=450)
fig.show()

fig2 = px.line(
    funnel.sort_values("date"),
    x="date",
    y="overall_conversion_rate",
    title="Overall Conversion Rate (30-day trend)",
    labels={"overall_conversion_rate": "Conversion Rate"},
)
fig2.update_yaxes(tickformat=".1%")
fig2.show()

## Cohort Retention

`mart_cohort_retention` contains week-over-week retention indexed by the user's signup cohort week. Values represent the fraction of cohort members who placed at least one order in each subsequent week.

In [ ]:
cohort = sf_query("""
    SELECT
        cohort_week,
        weeks_since_signup,
        retention_rate
    FROM mart_cohort_retention
    WHERE cohort_week >= DATEADD('week', -16, CURRENT_DATE)
    ORDER BY cohort_week, weeks_since_signup
""")

pivot = cohort.pivot(index="cohort_week", columns="weeks_since_signup", values="retention_rate")
pivot.index = pivot.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".0%",
    cmap="YlOrRd_r",
    linewidths=0.5,
    ax=ax,
    vmin=0,
    vmax=1,
    cbar_kws={"label": "Retention Rate"},
)
ax.set_title("Weekly Cohort Retention (last 16 cohorts)", fontsize=14, pad=12)
ax.set_xlabel("Weeks Since Signup")
ax.set_ylabel("Cohort Week")
plt.tight_layout()
plt.show()

## Product Performance

Product aggregations are written by the Spark `raw_to_curated` job directly to Delta Lake on S3/MinIO. We read them here via PyArrow dataset without going through Snowflake to demonstrate the lakehouse access pattern.

In [ ]:
import pyarrow.fs as pafs

minio_endpoint = os.environ.get("MINIO_ENDPOINT", "http://localhost:9000")
minio_user     = os.environ.get("MINIO_ROOT_USER", "minioadmin")
minio_password = os.environ.get("MINIO_ROOT_PASSWORD", "minioadmin")

s3_fs = pafs.S3FileSystem(
    access_key=minio_user,
    secret_key=minio_password,
    endpoint_override=minio_endpoint.replace("http://", "").replace("https://", ""),
    scheme="http",
)

product_ds = ds.dataset(
    "lakehouse-curated/product_performance",
    filesystem=s3_fs,
    format="parquet",
)

product_perf = product_ds.to_table().to_pandas()
product_perf["date"] = pd.to_datetime(product_perf["date"])

print(f"Loaded {len(product_perf):,} product-day rows")
display(product_perf.dtypes)
display(product_perf.head())

top20 = (
    product_perf
    .groupby("product_id")["total_views"]
    .sum()
    .nlargest(20)
    .reset_index()
    .rename(columns={"total_views": "cumulative_views"})
)
print("\nTop 20 products by cumulative views:")
display(top20)

In [ ]:
fig = px.bar(
    top20,
    x="cumulative_views",
    y="product_id",
    orientation="h",
    title="Top 20 Products by Total Views",
    labels={"cumulative_views": "Cumulative Views", "product_id": "Product ID"},
    color="cumulative_views",
    color_continuous_scale="Blues",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600, showlegend=False)
fig.show()

## User Segments

The `dim_users` table in Snowflake is populated by the dbt `dim_users` model which joins Snowflake mart data with user feature scores written by the Ray/Spark feature engineering job. RFM segments and LTV estimates are computed there.

In [ ]:
users = sf_query("""
    SELECT
        user_id,
        rfm_segment,
        ltv_estimate,
        total_orders,
        total_spend,
        days_since_last_order
    FROM dim_users
    WHERE rfm_segment IS NOT NULL
""")

print(f"Loaded {len(users):,} users with RFM segments")

segment_counts = users["rfm_segment"].value_counts().reset_index()
segment_counts.columns = ["segment", "count"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=segment_counts, x="count", y="segment", palette="viridis", ax=axes[0])
axes[0].set_title("User Count by RFM Segment")
axes[0].set_xlabel("User Count")
axes[0].set_ylabel("")

ltv_by_segment = users.groupby("rfm_segment")["ltv_estimate"].median().sort_values(ascending=False)
ltv_by_segment.plot(kind="bar", ax=axes[1], color=sns.color_palette("viridis", len(ltv_by_segment)))
axes[1].set_title("Median LTV Estimate by RFM Segment")
axes[1].set_xlabel("")
axes[1].set_ylabel("Median LTV ($)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

fig2 = px.histogram(
    users,
    x="ltv_estimate",
    color="rfm_segment",
    nbins=50,
    title="LTV Estimate Distribution by Segment",
    labels={"ltv_estimate": "LTV Estimate ($)"},
    barmode="overlay",
    opacity=0.7,
)
fig2.update_layout(height=450)
fig2.show()

print("\nLTV summary by segment:")
display(
    users.groupby("rfm_segment")["ltv_estimate"]
    .agg(["mean", "median", "std", "count"])
    .round(2)
    .sort_values("median", ascending=False)
)

conn.close()
print("\nSnowflake connection closed")